# Session 2: Structured Intelligence & Validation
Now we've got to the stage of being able to make a passable chatbot, we can start to take advantage of the real benefits of using PydanticAI that help integrate it's systems into larger works - output validation!

In this session we start using the result_type argument, implement logical guards with @field_validator, and understand the Retry Loop.

In [ ]:
import os
from datetime import datetime
from enum import Enum
from typing import List, Optional

from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    openai_api_key: str
    open_ai_default_model: str = "openai:gpt-5-nano"


settings = Settings()

## Part 1: Typed Outputs
First we started asking models "pretty please return JSON or I'll lose my job!", then we started providing JSON schemas as part of the model context. Now most LLMs providers are capable of going one step further and restricting model outputs to guarantee the output matches the schema - by actually restricting the output tokens.

In [ ]:
# 1. Define the Schema
class Genre(str, Enum):
    ACTION = "action"
    ADVENTURE = "adventure"
    COMEDY = "comedy"
    DRAMA = "drama"
    HORROR = "horror"
    ROMANCE = "romance"
    SCI_FI = "sci-fi"
    THRILLER = "thriller"
    FANTASY = "fantasy"
    DOCUMENTARY = "documentary"
    ANIMATION = "animation"
    CRIME = "crime"
    MYSTERY = "mystery"
    WAR = "war"
    WESTERN = "western"


class MovieExtraction(BaseModel):
    title: str
    director: str
    year: int
    genres: List[Genre]
    short_summary: str = Field(description="A one-sentence summary")


# 2. Define the Agent with the Schema
extraction_agent = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtraction,  # <--- The Magic
    system_prompt="Extract movie details from the user text.",
)

# 3. Run
text = "I just watched Inception by Nolan. It came out in 2010. Mind bending sci-fi thriller."
result = await extraction_agent.run(text)

# 4. Result is a validated Object, not a dict or string
print(f"Type: {type(result.output)}")
print(f"Movie: {result.output.title} ({result.output.year})")
print(f"Genres: {result.output.genres}")

## Part 2: Enforcing Logic: Validators as Guardrails
Getting the structure right is easy (JSON Mode). Getting the logic right is hard. What if a film review extractor hallucinates a year like 2030? Or returns a confidence score of 150%?

Pydantic to the rescue again. We use standard Pydantic @field_validator. If a validation error is raised, PydanticAI catches it, sends the error message back to the LLM, and asks it to try again.

The Retry Loop Visualized:  
* `Agent`: "Here is the data: {year: 1800}"
* Pydantic: ValidationError: Year must be > 1900
* PydanticAI: (Intercepts error) -> Sends user message: "Error: Year must be > 1900. Fix this."
* `Agent`: "Apologies. {year: 1900}" -> Success

In [ ]:
# 1. Define MovieExtractionV2 with validation
class MovieExtractionV2(BaseModel):
    title: str
    director: str
    year: Optional[int] = None
    genres: List[Genre]
    short_summary: str = Field(description="A one-sentence summary")

    @field_validator("year")
    @classmethod
    def validate_year_not_future(cls, v: Optional[int]) -> Optional[int]:
        """Ensure the film wasn't released in the future."""
        if v is None:
            return v
        current_year = datetime.now().year
        if v > current_year:
            raise ValueError(
                f"Year {v} is in the future. The current year is {current_year}. If the release year is an estimate in"
                "the future return None."
            )
        return v


# 2. Create agent with V2 schema
extraction_agent_v2 = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtractionV2,
    system_prompt="Extract movie details from the user text. Be accurate with release years.",
)

# 3. Test with a normal case (should work)
print("=== Test 1: Valid movie (past year) ===")
text1 = "I just watched The Matrix by the Wachowskis. It came out in 1999. Groundbreaking sci-fi action film."
result1 = await extraction_agent_v2.run(text1)
print(f"Movie: {result1.output.title} ({result1.output.year})")
print("Validation passed! ✓\n")

# 4. Test with a text that might cause hallucination (the validator will catch it)
print("=== Test 2: Potential future year hallucination ===")
print("Note: If the model tries to return a future year, the validator will catch it")
print("and PydanticAI will automatically retry with the error message.\n")

text2 = "I'm excited about the upcoming sequel that will be released in 2027."
result2 = await extraction_agent_v2.run(text2)
print(f"Movie: {result2.output.title} ({result2.output.year})")
print("Validation passed! ✓")

In [ ]:
from rich import print as rprint  # Alex - TIL you could do this!

# Let's inspect the messages list to verify it's just a list of objects
rprint(result2.all_messages())

## Part 3: Type Descriptors Guide LLM Behavior
PydanticAI doesn't just use your Pydantic models for validation - it also passes the field descriptions and type information directly to the LLM. This means your `Field(description=...)` annotations become part of the model's instructions, helping it understand what you want *before* it generates output.

**Key Insight:** Better field descriptions = Better outputs, fewer retries.

The LLM sees:
- Field names and types
- Field descriptions (from `Field(description=...)`)
- Enum values (for Enum fields)
- Required vs optional fields

This contextual information helps the model generate more accurate outputs on the first try. They are also important if there are multiple possible interpretations of certain fields.


In [ ]:
# Example 2: Rich descriptions (better guidance for LLM)
class MovieExtractionRich(BaseModel):
    title: str = Field(description="The official title of the film")
    director: str = Field(description="Full name of the primary director")
    year: int = Field(description="An estimate of the year the film is set in")
    genres: List[Genre] = Field(description="List of 1-3 primary genres that best categorize the film")
    short_summary: str = Field(
        description="A concise one-sentence summary (15-25 words) that captures the film's core premise without spoilers"
    )


agent_rich = Agent(
    settings.open_ai_default_model,
    output_type=MovieExtractionRich,
    system_prompt="Extract movie details from the user text.",
)

print("=== With Rich Descriptions ===")
text1 = "I just watched The Matrix by the Wachowskis. It came out in 1999. Groundbreaking sci-fi action film."
result_rich = await agent_rich.run(text1)
print(f"Summary: {result_rich.output.short_summary}")
print(f"Length: {len(result_rich.output.short_summary.split())} words")
print(f"Year: Set in roughly {result_rich.output.year}")
print("\nNotice how the rich description lets us guide the LLM to another interpretation of the year!")

## 🧪 Practical Exercise: "The Unstructured Data Cleaner"
**Goal:**  
You are processing a queue of raw customer support emails. We'd like you to convert messy text into a structured SupportTicket objects.

**Requirements:**  
Create a Pydantic SupportTicket object meeting the following spec
* customer_name: str (Capitalized)
* sentiment_score: float (0.0 to 1.0, where 0 is angry, 1 is happy)
* severity: str (Enum: 'Low', 'Medium', 'High')
* actionable_items: List[str] (Bullet points of what to do)
* If severity is 'High' AND sentiment_score is < 0.3, the actionable_items list cannot be empty.
    * If it is empty, raise a ModelRetry (or ValueError) instructing the LLM to "Infer actionable items based on the angry tone."

**Example:**  
> SUBJECT: URGENT!!!!  
> FROM: karen.smith@email.com
>  
> I am absolutely furious. I have been on hold for 4 hours.   
> Your system charged me double for the subscription and then locked me out of my account.  
> Fix this NOW or I am suing.

In [ ]:
messy_email = """
SUBJECT: URGENT!!!!
FROM: karen.smith@email.com

I am absolutely furious. I have been on hold for 4 hours. 
Your system charged me double for the subscription and then locked me out of my account.
Fix this NOW or I am suing.
"""

In [ ]:
class Severity(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"


class SupportTicket(BaseModel):
    customer_name: str
    sentiment_score: float = Field(description="0=Angry, 1=Happy")
    severity: Severity
    actionable_items: List[str]

    @model_validator(mode="after")
    def enforce_action_on_high_severity(self):
        # Check logic
        is_angry = self.sentiment_score < 0.3
        is_severe = self.severity == Severity.HIGH
        no_actions = len(self.actionable_items) == 0

        if is_angry and is_severe and no_actions:
            # Raise ModelRetry to guide the LLM explicitly
            raise ModelRetry(
                "The customer is angry and severity is High. "
                "You MUST provide a list of actionable items (e.g., 'Refund user', 'Reset password'). "
                "Do not return an empty list."
            )
        return self


cleaner_agent = Agent(
    settings.open_ai_default_model,
    output_type=SupportTicket,
    system_prompt="Process the support ticket. Be realistic about sentiment.",
)

# Run
result = await cleaner_agent.run(messy_email)

from rich import print as rprint

rprint(result.output)